In [ ]:
import re
import pandas as pd
import textwrap

In [ ]:
FILE = r"..\cleaned_notes_sections.feather"

df = pd.read_feather(FILE)

print(df.columns)
print(len(df))

In [ ]:
import textwrap

for i, row in df.iterrows():
    # Strip whitespace and find the key case-insensitively
    section_key = "BESLUIT"
    if section_key:
        section_text = str(row["sections"].get(section_key, ""))
        if "lage bloeddruk" in section_text:
            wrapped_text = textwrap.fill(section_text, width=80)
            print(
                f"{'='*80}\nNote {i} - {section_key.strip()}:\n{'-'*80}\n{wrapped_text}\n"
            )
    else:
        print(f"{'='*80}\nNote {i} - Medische antecedenten section not found.\n")

In [ ]:
for i, row in df.head(10).iterrows():
    print(f"Note {i} section keys: {list(row['sections'].keys())}")

# Whole notes

In [ ]:
for i, row in df.head(5).iterrows():
    wrapped_text = textwrap.fill(row["text"], width=80)
    print(f"{'='*80}\nNote {i}:\n{'-'*80}\n{wrapped_text}\n")

In [ ]:
import re


def clean_note(text):
    pattern = r"""
        Patiënt\s+\[LASTNAME\].*?   # Start: Patiënt [LASTNAME] ...
        Gegenereerd\ op.*?\d{2}:\d{2}   # End: Gegenereerd op ... HH:MM
    """

    return re.sub(pattern, "", text, flags=re.DOTALL | re.VERBOSE)


df["cleaned_note"] = df["text"].apply(clean_note)

In [ ]:
for i, row in df.head(5).iterrows():
    wrapped_text = textwrap.fill(row["cleaned_note"], width=80)
    print(f"{'='*80}\nNote {i}:\n{'-'*80}\n{wrapped_text}\n")

In [ ]:
import re

# Regex pattern for the header block
pattern = r"""
    Patiënt\s+\[LASTNAME\].*?   # Start: Patiënt [LASTNAME] ...
    Gegenereerd\ op.*?\d{2}:\d{2}   # End: Gegenereerd op ... HH:MM
"""


def check_header(text):
    match = re.search(pattern, text, flags=re.DOTALL | re.VERBOSE)
    if match:
        print("Header block found!")
        # Optional: show what was matched
        print(match.group())
        return True
    else:
        print("No header block found.")
        return False


# Example usage
note_text = """Patiënt [LASTNAME], [FIRSTNAME] (V) ([BIRTHDATE]) Nr. [PATIENTID]
[PERSOONSGEGEVENS] Document Brief - Ontslagbrief Auteur:
Opname   Tijdstip onderzoek: 12/09/2017 10:13 Tijdstip
wijziging: 12/10/2017 16:58   Pagina 1/7 Gegenereerd op 08/10/2020 10:15 door
GEMS
Rest of the note text here..."""

check_header(note_text)